# Hidden Poetic Schools — Run This Top to Bottom

Just click each cell and press **Shift+Enter**, in order. That's it.


In [ ]:
# 1. Install everything needed (only takes a while the first time)
!pip install -r requirements.txt


In [ ]:
# 2. Setup
import config as cfg
from src.utils import set_seed, setup_plot_style
set_seed(cfg.RANDOM_SEED)
setup_plot_style()
print("Setup done.")


In [ ]:
# 3. Scrape the poems from aldiwan.net (SLOW — hours. Safe to re-run if interrupted.)
from src import scraper
scraper.run(resume=True, limit_poets=None)


In [ ]:
# 4. Load the scraped data
from src.data_loader import load_corpus, corpus_statistics
df, poems_by_poet = load_corpus(cfg.DB_PATH, cfg.MIN_POEMS_PER_POET, cfg.MIN_VERSES_PER_POEM)
stats, poet_summary = corpus_statistics(df, poems_by_poet)
print(stats)


In [ ]:
# 5. Compute embeddings (fast on your GPU)
from src.embeddings import load_model, embed_poem_verse_average
model = load_model(cfg.SBERT_MODEL_NAME)
poet_embeddings = embed_poem_verse_average(model, poems_by_poet)
print(f"Embedded {len(poet_embeddings)} poets.")


In [ ]:
# 6. Cluster them into "schools"
from src.similarity import compute_similarity_matrix
from src.clustering import full_clustering_pipeline

poet_names, sim_matrix = compute_similarity_matrix(poet_embeddings)
results = full_clustering_pipeline(poet_embeddings, sim_matrix, poet_names, poems_by_poet, cfg)

print("Clusters found:", results["hdbscan_metrics"]["n_clusters"])
print("Silhouette score:", results["hdbscan_metrics"]["silhouette"])


In [ ]:
# 7. See the picture
from src import visualization as viz
import matplotlib.pyplot as plt
from PIL import Image

save_path = cfg.FIGURES_DIR / "umap_clusters.png"
viz.fig_umap_hdbscan(results["umap_2d"], results["hdbscan_labels"], poet_names, save_path)

plt.figure(figsize=(10, 9))
plt.imshow(Image.open(save_path))
plt.axis("off")
plt.show()


In [ ]:
# 8. Save the cluster assignments to a CSV you can open in Excel
import pandas as pd

cluster_df = pd.DataFrame({
    "poet_name": results["poet_names"],
    "cluster": results["hdbscan_labels"],
})
cluster_df.sort_values("cluster").to_csv(cfg.TABLES_DIR / "cluster_assignments.csv", index=False)
print("Saved to output/tables/cluster_assignments.csv")
cluster_df.sort_values("cluster")


## Done

You've got:
- A picture of the clusters (shown above, also saved in `output/figures/`)
- `output/tables/cluster_assignments.csv` — which poet is in which cluster

That's the core result. If you want the baseline comparisons or the extra
validation checks (stability test, randomization test) added back in, just
ask — kept out of this version to keep things simple.
